# Proposições Chunking

Visão Geral

Este código implementa o método de blocking de proposição, baseado em [pesquisa de Tony Chen, et al.](https://doi.org/10.48550/arXiv.2312.06648). O sistema decompõe o texto de entrada em proposições que são atômicas, factuais, auto-suficientes e de natureza concisa, codifica as proposições em uma vector store, que pode mais tarde ser usada para recuperação

### Componentes-chave
1. ** Document Chunking: ** Dividir um documento em peças gerenciáveis para análise.
2. ** Geração de Propostas:** Usando LLMs para quebrar chunks de documentos em proposições factuais e auto-suficientes.
3. ** Verificação da Qualidade da Proposta: ** Avaliando proposições geradas com base na precisão, clareza, integralidade e concisão.
4. **Embedding e Vector Store:** Incorporando ambas as proposições e chunks maiores do documento em uma vector store para recuperação eficiente.
5. **Retirada e Comparação: ** Testando o sistema de recuperação com diferentes tamanhos de consulta e comparando os resultados do modelo baseado em proposição com o modelo baseado em chunks maiores.

<img src="../images/proposition_chunking.svg" alt="Reliable-RAG" width="600">

Motivação

A motivação por trás das proposições método de chunks é construir um sistema que decompõe um documento de texto em proposições concisas, factuais para a recuperação de informações mais granular e precisa. O uso de proposições permite um melhor controle e melhor manuseio de consultas específicas, particularmente para extrair conhecimento de textos detalhados ou complexos. A comparação entre o uso de chunks de proposição menores e chunks de documento maiores visa avaliar a eficácia da recuperação de informação granular.

Detalhes do Método

1. ** Variáveis de Ambiente de Carga:** O código começa por carregar variáveis de ambiente (por exemplo, chaves API para o serviço LLM) para garantir que o sistema pode acessar os recursos necessários.

2. **Documento Chunking: **
- O documento de entrada é dividido em chunks menores (chunks) usando `RecursiveCharacterTextSplitter`. Isso garante que cada chunk é de tamanho gerenciável para o LLM processar.

3. ** Geração de Propostas:**
- Proposições são geradas a partir de cada chunk usando um LLM (neste caso, "llama-3.1-70b-versátil"). A produção é estruturada como uma lista de declarações factuais, auto-suficientes que podem ser entendidas sem contexto adicional.

4. ** Verificação de qualidade: **
- Um segundo LLM avalia a qualidade das proposições, pontuando-as em precisão, clareza, completude e concisão. As propostas que satisfaçam os limiares exigidos em todas as categorias são mantidas.

5. **Proposições de Emblagem: **
- Proposições que passam a verificação de qualidade são incorporadas em uma vector store usando o modelo `OllamaEmbeddings`. Isto permite a recuperação baseada em similaridade de proposições quando as consultas são feitas.

6. **Retirada e Comparação: **
- Dois sistemas de recuperação são construídos: um usando os chunks baseados em proposição e outro usando chunks de documento maiores. Ambos são testados com várias consultas para comparar seu desempenho e a precisão dos resultados retornados.

Benefícios

- **Granularidade:** Ao quebrar o documento em pequenas proposições factuais, o sistema permite uma recuperação altamente específica, facilitando a extração de respostas precisas de documentos grandes ou complexos.
- ** Garantia de qualidade: ** O uso de uma verificação de qualidade LLM garante que as proposições geradas atendam a padrões específicos, melhorando a confiabilidade das informações recuperadas.
- ** Flexibilidade na recuperação: ** A comparação entre a recuperação baseada em proposição e a recuperação baseada em chunks maiores permite avaliar os trade-offs entre granularidade e contexto mais amplo em resultados de busca.

Implementação

1. ** Geração de Propostas:** O LLM é usado em conjunto com um prompt personalizado para gerar declarações factuais dos chunks do documento.
2. ** Verificação de qualidade: ** As proposições geradas passam por um sistema de classificação que avalia precisão, clareza, completude e concisão.
3. ** Integração com o Vector Store:** As proposições são armazenadas em uma vector store FAISS após serem incorporadas usando um modelo de incorporação pré-treinado, permitindo uma busca e recuperação eficiente baseada em similaridade.
4. ** Teste de pesquisa:** Várias consultas de teste são feitas para as lojas vetoriais (baseadas em proposição e chunks maiores) para comparar o desempenho de recuperação.

Resumo

Este código apresenta um método robusto para quebrar um documento em proposições auto-suficientes usando LLMs. O sistema realiza uma verificação de qualidade de cada proposta, incorpora-as em uma vector store e recupera as informações mais relevantes com base em consultas do usuário. A capacidade de comparar proposições granulares com chunks de documentos maiores fornece insight sobre qual método produz resultados mais precisos ou úteis para diferentes tipos de consultas. A abordagem enfatiza a importância da geração e recuperação de propostas de alta qualidade para extração precisa de informações de documentos complexos.

# Instalação e Importações do Pacote
A célula abaixo instala todos os pacotes necessários para executar este notebook.
.

In [ ]:
# Install required packages
!pip install faiss-cpu langchain langchain-community python-dotenv

In [1]:
### LLMs
import os
from dotenv import load_dotenv

# Load environment variables from '.env' file
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY') # For LLM

### Documento de ensaio

In [2]:
sample_content = """Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.
Conventional Wisdom vs. Founder Mode
The essay argues that the traditional advice given to growing companies—hiring good people and giving them autonomy—often fails when applied to startups.
This approach, suitable for established companies, can be detrimental to startups where the founder's vision and direct involvement are crucial. "Founder Mode" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional "manager mode" often advised by business schools and professional managers.
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's vision and culture.
Graham suggests that founders should leverage these strengths rather than conform to traditional managerial practices. "Founder Mode" is an emerging paradigm that is not yet fully understood or documented, with Graham hoping that over time, it will become as well-understood as the traditional manager mode, allowing founders to maintain their unique approach even as their companies scale.
Challenges of Scaling Startups
As startups grow, there is a common belief that they must transition to a more structured managerial approach. However, many founders have found this transition problematic, as it often leads to a loss of the innovative and agile spirit that drove the startup's initial success.
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's "Founder Mode" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart
. This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale.
"""

## Chunking

In [3]:
### Build Index
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

# Set embeddings
embedding_model = OllamaEmbeddings(model='nomic-embed-text:v1.5', show_progress=True)

# docs
docs_list = [Document(page_content=sample_content, metadata={"Title": "Paul Graham's Founder Mode Essay", "Source": "https://www.perplexity.ai/page/paul-graham-s-founder-mode-ess-t9TCyvkqRiyMQJWsHr0fnQ"})]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200, chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)

In [4]:
for i, doc in enumerate(doc_splits):
    doc.metadata['chunk_id'] = i+1 ### adding chunk id

Gerar Proposições

In [5]:
from typing import List
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_groq import ChatGroq

# Data model
class GeneratePropositions(BaseModel):
    """List of all the propositions in a given document"""

    propositions: List[str] = Field(
        description="List of propositions (factual, self-contained, and concise information)"
    )


# LLM with function call
llm = ChatGroq(model="llama-3.1-70b-versatile", temperature=0)
structured_llm= llm.with_structured_output(GeneratePropositions)

# Few shot prompting --- We can add more examples to make it good
proposition_examples = [
    {"document": 
        "In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.", 
     "propositions": 
        "['Neil Armstrong was an astronaut.', 'Neil Armstrong walked on the Moon in 1969.', 'Neil Armstrong was the first person to walk on the Moon.', 'Neil Armstrong walked on the Moon during the Apollo 11 mission.', 'The Apollo 11 mission occurred in 1969.']"
    },
]

example_proposition_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{document}"),
        ("ai", "{propositions}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_proposition_prompt,
    examples = proposition_examples,
)

# Prompt
system = """Please break down the following text into simple, self-contained propositions. Ensure that each proposition meets the following criteria:

    1. Express a Single Fact: Each proposition should state one specific fact or claim.
    2. Be Understandable Without Context: The proposition should be self-contained, meaning it can be understood without needing additional context.
    3. Use Full Names, Not Pronouns: Avoid pronouns or ambiguous references; use full entity names.
    4. Include Relevant Dates/Qualifiers: If applicable, include necessary dates, times, and qualifiers to make the fact precise.
    5. Contain One Subject-Predicate Relationship: Focus on a single subject and its corresponding action or attribute, without conjunctions or multiple clauses."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        few_shot_prompt,
        ("human", "{document}"),
    ]
)

proposition_generator = prompt | structured_llm

In [6]:
propositions = [] # Store all the propositions from the document

for i in range(len(doc_splits)):
    response = proposition_generator.invoke({"document": doc_splits[i].page_content}) # Creating proposition
    for proposition in response.propositions:
        propositions.append(Document(page_content=proposition, metadata={"Title": "Paul Graham's Founder Mode Essay", "Source": "https://www.perplexity.ai/page/paul-graham-s-founder-mode-ess-t9TCyvkqRiyMQJWsHr0fnQ", "chunk_id": i+1}))

## Verificação de Qualidade

In [7]:
# Data model
class GradePropositions(BaseModel):
    """Grade a given proposition on accuracy, clarity, completeness, and conciseness"""

    accuracy: int = Field(
        description="Rate from 1-10 based on how well the proposition reflects the original text."
    )
    
    clarity: int = Field(
        description="Rate from 1-10 based on how easy it is to understand the proposition without additional context."
    )

    completeness: int = Field(
        description="Rate from 1-10 based on whether the proposition includes necessary details (e.g., dates, qualifiers)."
    )

    conciseness: int = Field(
        description="Rate from 1-10 based on whether the proposition is concise without losing important information."
    )

# LLM with function call
llm = ChatGroq(model="llama-3.1-70b-versatile", temperature=0)
structured_llm= llm.with_structured_output(GradePropositions)

# Prompt
evaluation_prompt_template = """
Please evaluate the following proposition based on the criteria below:
- **Accuracy**: Rate from 1-10 based on how well the proposition reflects the original text.
- **Clarity**: Rate from 1-10 based on how easy it is to understand the proposition without additional context.
- **Completeness**: Rate from 1-10 based on whether the proposition includes necessary details (e.g., dates, qualifiers).
- **Conciseness**: Rate from 1-10 based on whether the proposition is concise without losing important information.

Example:
Docs: In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.

Propositons_1: Neil Armstrong was an astronaut.
Evaluation_1: "accuracy": 10, "clarity": 10, "completeness": 10, "conciseness": 10

Propositons_2: Neil Armstrong walked on the Moon in 1969.
Evaluation_3: "accuracy": 10, "clarity": 10, "completeness": 10, "conciseness": 10

Propositons_3: Neil Armstrong was the first person to walk on the Moon.
Evaluation_3: "accuracy": 10, "clarity": 10, "completeness": 10, "conciseness": 10

Propositons_4: Neil Armstrong walked on the Moon during the Apollo 11 mission.
Evaluation_4: "accuracy": 10, "clarity": 10, "completeness": 10, "conciseness": 10

Propositons_5: The Apollo 11 mission occurred in 1969.
Evaluation_5: "accuracy": 10, "clarity": 10, "completeness": 10, "conciseness": 10

Format:
Proposition: "{proposition}"
Original Text: "{original_text}"
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", evaluation_prompt_template),
        ("human", "{proposition}, {original_text}"),
    ]
)

proposition_evaluator = prompt | structured_llm

In [8]:
# Define evaluation categories and thresholds
evaluation_categories = ["accuracy", "clarity", "completeness", "conciseness"]
thresholds = {"accuracy": 7, "clarity": 7, "completeness": 7, "conciseness": 7}

# Function to evaluate proposition
def evaluate_proposition(proposition, original_text):
    response = proposition_evaluator.invoke({"proposition": proposition, "original_text": original_text})
    
    # Parse the response to extract scores
    scores = {"accuracy": response.accuracy, "clarity": response.clarity, "completeness": response.completeness, "conciseness": response.conciseness}  # Implement function to extract scores from the LLM response
    return scores

# Check if the proposition passes the quality check
def passes_quality_check(scores):
    for category, score in scores.items():
        if score < thresholds[category]:
            return False
    return True

evaluated_propositions = [] # Store all the propositions from the document

# Loop through generated propositions and evaluate them
for idx, proposition in enumerate(propositions):
    scores = evaluate_proposition(proposition.page_content, doc_splits[proposition.metadata['chunk_id'] - 1].page_content)
    if passes_quality_check(scores):
        # Proposition passes quality check, keep it
        evaluated_propositions.append(proposition)
    else:
        # Proposition fails, discard or flag for further review
        print(f"{idx+1}) Propostion: {proposition.page_content} \n Scores: {scores}")
        print("Fail")

17) Propostion: Startups often transition to a more structured managerial approach as they grow. 
 Scores: {'accuracy': 8, 'clarity': 9, 'completeness': 6, 'conciseness': 8}
Fail
31) Propostion: Delegating responsibilities to professional managers is not always the best approach as companies scale. 
 Scores: {'accuracy': 10, 'clarity': 10, 'completeness': 8, 'conciseness': 6}
Fail


### Proposições incorporadas em um vetorstore

In [9]:
# Add to vectorstore
vectorstore_propositions = FAISS.from_documents(evaluated_propositions, embedding_model)
retriever_propositions = vectorstore_propositions.as_retriever(
                search_type="similarity",
                search_kwargs={'k': 4}, # number of documents to retrieve
            )

OllamaEmbeddings: 100%|██████████| 29/29 [00:08<00:00,  3.62it/s]


In [10]:
query = "Who's management approach served as inspiartion for Brian Chesky's \"Founder Mode\" at Airbnb?"
res_proposition = retriever_propositions.invoke(query)

OllamaEmbeddings: 100%|██████████| 1/1 [00:00<00:00,  5.39it/s]


In [11]:
for i, r in enumerate(res_proposition):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Brian Chesky was advised to run Airbnb in a traditional managerial style. --- Chunk_id: 3
2) Content: Brian Chesky adopted a different approach to running Airbnb. --- Chunk_id: 3
3) Content: Brian Chesky is a co-founder of Airbnb. --- Chunk_id: 3
4) Content: Steve Jobs' management style at Apple influenced Brian Chesky's approach. --- Chunk_id: 3


### Comparando desempenho com tamanho de chunks maiores

In [12]:
# Add to vectorstore_larger_
vectorstore_larger = FAISS.from_documents(doc_splits, embedding_model)
retriever_larger = vectorstore_larger.as_retriever(
                search_type="similarity",
                search_kwargs={'k': 4}, # number of documents to retrieve
            )

OllamaEmbeddings: 100%|██████████| 3/3 [00:00<00:00,  5.35it/s]


In [13]:
res_larger = retriever_larger.invoke(query)

OllamaEmbeddings: 100%|██████████| 1/1 [00:00<00:00,  6.64it/s]


In [14]:
for i, r in enumerate(res_larger):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's "Founder Mode" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart
. This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale. --- Chunk_id: 3
2) Content: Unique Founder Abil

Teste

#### Teste - 1

In [15]:
test_query_1 = "what is the essay \"Founder Mode\" about?"
res_proposition = retriever_propositions.invoke(test_query_1)
res_larger = retriever_larger.invoke(test_query_1)

OllamaEmbeddings: 100%|██████████| 1/1 [00:00<00:00,  8.06it/s]


In [16]:
for i, r in enumerate(res_proposition):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Founder Mode is an emerging paradigm that is not yet fully understood or documented. --- Chunk_id: 2
2) Content: Founder Mode is not yet fully understood or documented. --- Chunk_id: 1
3) Content: Founder Mode is an emerging paradigm. --- Chunk_id: 1
4) Content: Paul Graham's essay 'Founder Mode' challenges conventional wisdom about scaling startups. --- Chunk_id: 1


In [17]:
for i, r in enumerate(res_larger):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.
Conventional Wisdom vs. Founder Mode
The essay argues that the traditional advice given to growing companies—hiring good people and giving them autonomy—often fails when applied to startups.
This approach, suitable for established companies, can be detrimental to startups where the founder's vision and direct involvement are crucial. "Founder Mode" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional "manager mode" often advised by business schools and professional managers.
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's v

#### Teste - 2

In [18]:
test_query_2 = "who is the co-founder of Airbnb?"
res_proposition = retriever_propositions.invoke(test_query_2)
res_larger = retriever_larger.invoke(test_query_2)

OllamaEmbeddings: 100%|██████████| 1/1 [00:00<00:00, 15.18it/s]


In [19]:
for i, r in enumerate(res_proposition):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Brian Chesky is a co-founder of Airbnb. --- Chunk_id: 3
2) Content: Brian Chesky adopted a different approach to running Airbnb. --- Chunk_id: 3
3) Content: Brian Chesky was advised to run Airbnb in a traditional managerial style. --- Chunk_id: 3
4) Content: Running Airbnb in a traditional managerial style led to poor outcomes. --- Chunk_id: 3


In [20]:
for i, r in enumerate(res_larger):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's "Founder Mode" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart
. This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale. --- Chunk_id: 3
2) Content: Paul Graham's essay

#### Teste - 3

In [21]:
test_query_3 = "when was the essay \"founder mode\" published?"
res_proposition = retriever_propositions.invoke(test_query_3)
res_larger = retriever_larger.invoke(test_query_3)

OllamaEmbeddings: 100%|██████████| 1/1 [00:00<00:00,  7.71it/s]


In [22]:
for i, r in enumerate(res_proposition):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Paul Graham published an essay called 'Founder Mode' in September 2024. --- Chunk_id: 1
2) Content: Founder Mode is an emerging paradigm. --- Chunk_id: 1
3) Content: Founder Mode is an emerging paradigm that is not yet fully understood or documented. --- Chunk_id: 2
4) Content: Founder Mode is not yet fully understood or documented. --- Chunk_id: 1


In [23]:
for i, r in enumerate(res_larger):
    print(f"{i+1}) Content: {r.page_content} --- Chunk_id: {r.metadata['chunk_id']}")

1) Content: Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.
Conventional Wisdom vs. Founder Mode
The essay argues that the traditional advice given to growing companies—hiring good people and giving them autonomy—often fails when applied to startups.
This approach, suitable for established companies, can be detrimental to startups where the founder's vision and direct involvement are crucial. "Founder Mode" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional "manager mode" often advised by business schools and professional managers.
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's v

Comparação

* * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
-------------------------------------------------------------------
• **Precisão em Resposta ** Alta: Fornece respostas focadas e diretas. Médio: Fornece mais contexto, mas pode incluir informações irrelevantes. □
"Clarity and Brevity"** "Alta: Clara e concisa, evita detalhes desnecessários. Médio: Mais abrangente, mas pode ser esmagador. □
* Riqueza contextual** Pode faltar contexto, com foco em proposições específicas. Alta: Fornece contexto adicional e detalhes. □
* Compreensão** Baixa: Pode omitir um contexto mais amplo ou detalhes suplementares. Alta: Oferece uma visão mais completa com informações extensas. □
* Fluxo narrativo** Média: Pode ser fragmentado ou desarticulado. Alta: Preserva o fluxo lógico e a coerência do documento original. □
Sobrecarga de Informação** Baixa: Menos provável de sobrecarregar com excesso de informação. Alto: Risco de esmagar o usuário com muita informação. □
Use a Adequação de Casos** Melhor para consultas rápidas e factuais. Melhor para consultas complexas que exigem compreensão aprofundada. □
Eficiência** Alta: Fornece respostas rápidas e direcionadas. □ Meio: Pode exigir mais esforço para peneirar o conteúdo adicional. □
□ **Especificidade** □ Alta: Respostas precisas e direcionadas. • Meio: As respostas podem ser menos direcionadas devido à inclusão de contexto mais amplo. □


![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--proposition-chunking)